<a href="https://colab.research.google.com/github/3295354473-art/5fold-cv-project/blob/main/NGAFID_DATASET_MINIROCKET_EXAMPLE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!python --version

Python 3.12.13


In [14]:
!git clone https://github.com/hyang0129/NGAFIDDATASET.git

!(cd NGAFIDDATASET ; git checkout main; git reset --hard HEAD; git pull)
!(cd NGAFIDDATASET ; pip install -r requirements.txt -q)

!pip install tsai -q

fatal: destination path 'NGAFIDDATASET' already exists and is not an empty directory.
Already on 'main'
Your branch is up to date with 'origin/main'.
HEAD is now at fa72386 Update README.md
Already up to date.


In [15]:
pip install ipython==7.34.0 decorator>=4.3.0 --quiet

In [16]:
import sys
sys.path.append('/content/NGAFIDDATASET')

In [17]:
from ngafiddataset.dataset.dataset import NGAFID_Dataset_Manager
from ngafiddataset.utils import connect_to_tpu

from tsai.basics import *
from tsai.models.MINIROCKET_Pytorch import *
from tsai.models.utils import *
import pandas as pd


# strategy = connect_to_tpu()

/content/NGAFIDDATASET/ngafiddataset/dataset/dataset.py:53: SyntaxWarning: invalid escape sequence '\o'
  logger.info('Downloading and extracting Parquet Files to %s\one_parq.  Please open them using dask dataframes' % destination)
/content/NGAFIDDATASET/ngafiddataset/dataset/dataset.py:6: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


### Mount Google Drive

First, we'll mount your Google Drive so you can access files stored there.

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Specify Dataset Path on Google Drive

Please enter the full path to your `2days.zip` file *within your Google Drive* in the code cell below. For example, if it's directly in 'My Drive', the path might be `/content/drive/MyDrive/2days.zip`. If it's in a subfolder like 'My Drive/Datasets/', it would be `/content/drive/MyDrive/Datasets/2days.zip`.

In [22]:
#@title Enter the path to 2days.zip on your Google Drive
gdrive_zip_path = "/content/drive/MyDrive/NGAFIDDATASET/2days.tar" #@param {type:"string"}

### Copy and Unzip the Dataset

Now, we'll copy the `2days.zip` file from your Google Drive to the Colab environment's `/content/` directory and then unzip it. This will create the `/content/2days/` directory that `NGAFID_Dataset_Manager` expects.

In [23]:
import shutil
import os

# Define the destination directory for the extracted content
local_dataset_dir = '/content/2days'

# Ensure the destination directory exists if it's meant to be created by the extraction tool
# If it's expected to be present, and it's not, the DatasetManager will download.
# So, we ensure the tar is where it expects it to be unpacked.

# First, copy the tar file to a location where it can be easily extracted
# We'll copy it to /content/ so that the extracted contents also go to /content/2days
print(f"Copying {gdrive_zip_path} to /content/2days.tar...")
shutil.copyfile(gdrive_zip_path, '/content/2days.tar')
print("Copy complete.")

# Extract the file into /content/
print(f"Extracting /content/2days.tar to {local_dataset_dir}...")
!tar -xf /content/2days.tar -C /content/
print("Extraction complete.")

# Verify the existence of the expected directory
if os.path.exists(local_dataset_dir):
    print(f"Dataset directory '{local_dataset_dir}' successfully created.")
else:
    print(f"Error: Dataset directory '{local_dataset_dir}' was not created.")

Copying /content/drive/MyDrive/NGAFIDDATASET/2days.tar to /content/2days.tar...
Copy complete.
Extracting /content/2days.tar to /content/2days...
Extraction complete.
Dataset directory '/content/2days' successfully created.


In [27]:
import os
import importlib
import ngafiddataset.dataset.dataset as dataset_module

# Path to the library file that needs modification
file_path = '/content/NGAFIDDATASET/ngafiddataset/dataset/dataset.py'

if os.path.exists(file_path):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    # The problematic line is line 66 (1-indexed), which is index 65 (0-indexed)
    problematic_line_index = 65

    # Check if the line exists and contains the expected download call
    if problematic_line_index < len(lines) and "self.download(name, destination, extract)" in lines[problematic_line_index]:
        # Comment out the line to prevent unconditional download
        lines[problematic_line_index] = "# " + lines[problematic_line_index]
        with open(file_path, 'w') as f:
            f.writelines(lines)
        print(f"Successfully commented out line {problematic_line_index + 1} in {file_path} to prevent unconditional download.")
    else:
        print(f"Warning: Line {problematic_line_index + 1} not found or doesn't match expected content in {file_path}. Manual verification might be needed.")
else:
    print(f"Error: Library file not found at {file_path}")

# Reload the module to apply changes
importlib.reload(dataset_module)

# Now, re-import NGAFID_Dataset_Manager and proceed
from ngafiddataset.dataset.dataset import NGAFID_Dataset_Manager

dm = NGAFID_Dataset_Manager('2days', destination='/content/')
df = pd.read_csv('/content/2days/flight_header.csv', index_col = 'Master Index')

dm.data_dict =  dm.construct_data_dictionary(numpy=True)

Successfully commented out line 66 in /content/NGAFIDDATASET/ngafiddataset/dataset/dataset.py to prevent unconditional download.


/content/NGAFIDDATASET/ngafiddataset/dataset/dataset.py:53: SyntaxWarning: invalid escape sequence '\o'
  logger.info('Downloading and extracting Parquet Files to %s\one_parq.  Please open them using dask dataframes' % destination)


  0%|          | 0/11446 [00:00<?, ?it/s]

In [28]:
df.head(5)

,before_after,date_diff,flight_length,label,hierarchy,fold,class,target_class,hclass,number_flights_before
Master Index,,,,,,,,,,
1,1,-1,4723.0,intake gasket leak/damage,NaN,0,10,10,0,0
2,1,-2,4649.0,intake gasket leak/damage,NaN,0,10,10,0,3
7,0,1,3482.0,intake gasket leak/damage,NaN,0,10,0,0,-1
9,1,-1,4979.0,intake gasket leak/damage,NaN,0,10,10,0,0
11,0,2,5204.0,intake gasket leak/damage,NaN,0,10,0,0,-1


In [30]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=df)

https://docs.google.com/spreadsheets/d/1Y58YuzkhPkA7yTs121Z3l45C1V0MUOsTVUqzNdB4YS0/edit#gid=0


In [31]:
number_classes = len(dm.flight_header_df['class'].unique())
number_classes

number_hierarchies = len(dm.flight_header_df['hclass'].unique())
number_hierarchies


5

In [32]:
from fastai.vision import *

In [33]:

from fastai.callback.progress import CSVLogger

# from fastai.callbacks import CSVLogger
from tqdm.autonotebook import tqdm

In [ ]:
import gc # Import garbage collector

save_path = '.'
model_name = 'MINIROCKET_MULTI'


for fold in tqdm(range(5)):

    mode = 'before_after'
    mode = 'classes'

    save_filename = save_path + '%s_%i' % (model_name, fold)

    train_dict = dm.get_numpy_dataset(fold = fold, training = True)
    test_dict = dm.get_numpy_dataset(fold = fold, training  = False)

    train_X = np.array(train_dict['data'], dtype = np.float32)

    # Delete original dicts to free memory
    del train_dict
    gc.collect()

    train_X = (train_X - dm.mins)/(dm.maxs - dm.mins)
    train_X = np.nan_to_num(train_X, copy = False)

    test_X = np.array(test_dict['data'], dtype = np.float32)

    # Delete original dicts to free memory
    del test_dict
    gc.collect()

    test_X = (test_X- dm.mins)/(dm.maxs - dm.mins)
    test_X = np.nan_to_num(test_X, copy = False)

    train_Y = np.array(train_dict['target_class'])
    test_Y = np.array(test_dict['target_class'])

    # train_Y = np.array(train_dict['before_after'])
    # test_Y = np.array(test_dict['before_after'])

    splits = [list(np.arange(len(train_Y)))]

    splits.append(list(np.arange(len(test_Y) )+ len(train_Y)))


    torch.cuda.empty_cache()
    mrf = MiniRocketFeatures(train_X.shape[1], train_X.shape[2]).to(default_device())
    chunksize = 32 # Reduced chunksize to save memory

    mrf.fit(train_X, chunksize = chunksize)

    # Delete train_X to free memory before concatenating
    del train_X
    gc.collect()

    X_feat = get_minirocket_features(np.concatenate([test_X]), mrf, chunksize=chunksize, to_np=True) # Only concatenate test_X since train_X is deleted

    # Delete test_X to free memory
    del test_X
    gc.collect()

    Y = np.concatenate([train_Y, test_Y])

    # Delete train_Y and test_Y to free memory
    del train_Y, test_Y
    gc.collect()

    PATH = Path("./models/MRF.pt")
    PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(mrf.state_dict(), PATH)

    tfms = [None, TSClassification()]
    batch_tfms = TSStandardize(by_sample=True)
    dls = get_ts_dls(X_feat, Y , splits = splits, tfms=tfms, batch_tfms=batch_tfms)
    model = build_ts_model(MiniRocketHead, dls=dls)

    # Delete X_feat and Y to free memory once dls is created
    del X_feat, Y
    gc.collect()

    learn = Learner(dls, model, metrics=accuracy, cbs=[ShowGraph(), CSVLogger(save_filename)])

    results = learn.fit_one_cycle(200, 2.5e-5)

    # Delete learner and model to free memory for the next fold
    del learn, model, dls
    gc.collect()

  0%|          | 0/5 [00:00<?, ?it/s]